# Report window

> Display a session-usage report in a Tkinter window.

This module renders report data produced elsewhere in the application as a small, table-based desktop dashboard. It displays session summary values, foreground-app durations, and—when enabled—window-title details for the selected app.

The module is presentation-only: it expects a prepared `report_data` structure and does not query the database or calculate report aggregates. It is used by the tray/controller layer to show completed-session reports or current-session statistics.


In [ ]:
#| default_exp report_window

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import tkinter as tk
from tkinter import ttk

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import snooper_pkg.config as cf

## Duration formatting

In [ ]:
#| export
def fmt_duration(seconds):
    """Format a duration in seconds as whole minutes and seconds."""
    return f"{seconds // 60}m {seconds % 60}s"

## Application summary table

In [ ]:
#| export
def show_apps(root, report_data):
    """Create and populate the foreground-application summary table."""
    tree = ttk.Treeview(root, columns=("time", "percent"))

    tree.heading("#0", text="App")
    tree.heading("time", text="Time")
    tree.heading("percent", text="% Active")

    for i, event in enumerate(report_data["events"]):
        tree.insert("", "end", iid=str(i),
                     text=event['app'],
                       values=(f"{fmt_duration(event['duration_seconds'])}",
                                f"{event['percent_active']}%"))
    tree.pack(fill="both", expand=True)
    return tree

## Session summary

In [ ]:
#| export
def show_summary(root, report_data):
    """Add the session summary fields to the report window."""
    session = report_data["session"]
    summary = report_data["summary"]

    frame = ttk.LabelFrame(root, text="Session Summary")
    frame.pack(fill="x", padx=10, pady=10)

    ttk.Label(frame, text=f"Start: {session['start_time']}").grid(row=0, column=0, sticky="w", padx=8, pady=3)
    ttk.Label(frame, text=f"End: {session['end_time']}").grid(row=0, column=1, sticky="w", padx=8, pady=3)
    ttk.Label(frame, text=f"Total: {fmt_duration(session['duration_seconds'])}").grid(row=0, column=2, sticky="w", padx=8, pady=3)

    ttk.Label(frame, text=f"Active: {fmt_duration(summary['active_duration_seconds'])} ({summary['active_percent']}%)").grid(row=1, column=0, sticky="w", padx=8, pady=3)
    ttk.Label(frame, text=f"Idle: {fmt_duration(summary['idle_duration_seconds'])} ({summary['idle_percent']}%)").grid(row=1, column=1, sticky="w", padx=8, pady=3)
    ttk.Label(frame, text=f"Apps: {summary['distinct_app_count']}").grid(row=1, column=2, sticky="w", padx=8, pady=3)

    ttk.Label(frame, text=f"Top app: {summary['top_app']}").grid(row=2, column=0, columnspan=3, sticky="w", padx=8, pady=3)

## Selected-app title details

In [ ]:
#| export
def show_titles(root):
    """Create the table used to show titles for the selected application."""
    frame = ttk.LabelFrame(root, text="Titles for selected app")
    frame.pack(fill="both", expand=True, padx=10, pady=10)

    tree = ttk.Treeview(frame, columns=("time",))
    tree.heading("#0", text="Title")
    tree.heading("time", text="Time")
    tree.pack(fill="both", expand=True)

    return tree

def update_titles(title_tree, report_data, app_tree):
    """Replace title-table rows with title durations for the selected app."""
    for row in title_tree.get_children():
        title_tree.delete(row)
    selection = app_tree.selection()
    if not selection: return
    selected_iid = selection[0]
    selected_event = report_data["events"][int(selected_iid)]   
    for title_info in selected_event["top_titles"]:
        title_tree.insert(
            "",
            "end",
            text=title_info["title"],
            values=(fmt_duration(title_info["duration_seconds"]),)
        )

## Report window lifecycle

In [ ]:
#| export
def show_report(
    parent,
    report_data,
    report_title="Usage Report",
    is_show_titles=cf.SHOW_WINDOW_TITLES,
):
    """Open a report as a child of the application's Tk root."""
    window = tk.Toplevel(parent)
    window.title(report_title)

    show_summary(window, report_data)
    app_tree = show_apps(window, report_data)

    if is_show_titles:
        title_tree = show_titles(window)
        rows = app_tree.get_children()

        if rows:
            app_tree.selection_set(rows[0])
            update_titles(title_tree, report_data, app_tree)

        app_tree.bind(
            "<<TreeviewSelect>>",
            lambda event: update_titles(title_tree, report_data, app_tree),
        )

NameError: name 'cf' is not defined

1. Report-data shape: this module treats report_data["events"] as a list of app-summary dictionaries. The project context describes a preferred future structure where "events" is a dictionary containing "top_apps", "titles_by_app", and "timeline". Confirm the reporter’s current return shape before documenting that structure more specifically.

2. Title-data availability: update_titles assumes every app event has a "top_titles" key, including any idle event. Confirm whether idle rows include that key or whether they should be excluded from title-detail display.

3. Configuration default timing: is_show_titles = cf.SHOW_WINDOW_TITLES is evaluated when the function is defined, rather than when it is called. That is valid for a static configuration setting, but it will not reflect changes to cf.SHOW_WINDOW_TITLES made later in the same process.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()